# Mesclagem RF e GB

Combina as previsões fora-da-dobra gravadas por `treino_rf_gb_por_area.ipynb` por média ponderada dos dois modelos.

In [ ]:
# == caminhos do projeto ==
# A pasta deste caderno, dentro de notebooks/. Os CSVs sao lidos e gravados
# nela; as leituras entre pastas usam ../<outra pasta>. Para que esses caminhos
# valham qualquer que seja o diretorio em que o kernel foi aberto, o bloco
# abaixo localiza a raiz do repositorio -- a pasta que contem notebooks/ e
# documents/ -- e entra na pasta do caderno. No Colab, onde nao existe
# repositorio e os arquivos ficam soltos ao lado do caderno, nada e alterado.
import os

PASTA_CADERNO = "colabs_S4/testes_extras/testes_treinamento"


def raiz_repositorio(inicio=None):
    """Sobe a arvore de diretorios ate a raiz do repositorio; None se nao achar."""
    atual = os.path.abspath(inicio or os.getcwd())
    while True:
        if all(os.path.isdir(os.path.join(atual, p)) for p in ("notebooks", "documents")):
            return atual
        pai = os.path.dirname(atual)
        if pai == atual:
            return None
        atual = pai


RAIZ_REPO = raiz_repositorio()
if RAIZ_REPO:
    os.chdir(os.path.join(RAIZ_REPO, "notebooks", PASTA_CADERNO))
print("Diretorio de trabalho:", os.path.abspath("."))

In [ ]:
import os

import pandas as pd
import numpy as np

PASTA_DADOS = "colabs_S3"

BASES = [
    ".", "..", os.path.join("..", ".."), os.path.join("..", "..", ".."), "/content",
    "notebooks", os.path.join("..", "notebooks"),
    os.path.join("notebooks", PASTA_DADOS),
]

def resolve(pasta, arquivo):
    tentativas = []
    # A pasta declarada tem prioridade sobre o arquivo solto: sem isso, um
    # homonimo numa pasta vizinha pode ser lido no lugar do arquivo pedido.
    for partes in ((pasta, arquivo), (arquivo,)):
        for base in BASES:
            caminho = os.path.join(base, *partes)
            tentativas.append(caminho)
            if os.path.exists(caminho):
                return caminho
    raise FileNotFoundError(
        "Arquivo nao encontrado: " + arquivo
        + ". Diretorio atual: " + os.path.abspath(".")
        + ". Caminhos tentados: " + "; ".join(tentativas)
    )

In [ ]:
previsoes = pd.read_csv(resolve(PASTA_DADOS, "previsoes_rf_gb.csv"))
limites = pd.read_csv(resolve(PASTA_DADOS, "limites_ativos.csv")).set_index("tc_name")
previsoes.shape

In [ ]:
MARGEM_KPI = 0.10   # margem de erro acordada no TAPI


def _valida(y_real, y_previsto):
    """Converte para vetores e recusa entradas que produziriam metrica sem sentido.

    A conversao para numpy e necessaria porque a subtracao entre duas Series do pandas
    alinha pelos indices, e nao pela posicao. Uma Series com indice original e outra
    com indice reiniciado produziriam diferencas entre pares errados, em silencio.

    O alvo precisa ser estritamente positivo. A restricao e legitima neste projeto,
    pois trata-se de temperatura absoluta em graus Celsius, cujo minimo na base e de
    27,69 graus. Ela existe para transformar em erro visivel o engano de passar valores
    padronizados, caso em que o alvo se aproximaria de zero e o erro percentual
    explodiria em vez de acusar o problema.
    """
    y_real = np.asarray(y_real, dtype=float).ravel()
    y_previsto = np.asarray(y_previsto, dtype=float).ravel()

    if y_real.shape != y_previsto.shape:
        raise ValueError(f"dimensoes diferentes: {y_real.shape} e {y_previsto.shape}")
    if y_real.size == 0:
        raise ValueError("nenhuma observacao a avaliar")
    if np.isnan(y_real).any():
        raise ValueError("ha valores ausentes em y_real; a mascara deve ser aplicada antes")
    if np.isnan(y_previsto).any():
        raise ValueError("ha valores ausentes em y_previsto")
    if (y_real <= 0).any():
        raise ValueError(
            "y_real contem valor nao positivo; o erro percentual exige temperatura "
            "absoluta em graus Celsius, e nao valores padronizados")

    return y_real, y_previsto


def avaliar(y_real, y_previsto, limite=None):
    """Metricas de erro de um componente, com o alvo em graus Celsius.

    Sempre devolvidas:

    n            numero de observacoes que sustentam as metricas
    MAE          erro absoluto medio, na unidade do alvo
    RMSE         raiz do erro quadratico medio, na mesma unidade
    R2           fracao da variancia do alvo explicada pelo modelo
    vies         erro medio com sinal, positivo quando o modelo superestima
    RMSE/MAE     razao entre as duas, indicando concentracao dos erros
    MAPE_%       erro percentual absoluto medio
    WAPE_%       erro absoluto total dividido pela soma dos alvos
    dentro_10_%  proporcao de predicoes dentro da margem de 10 por cento do TAPI

    Devolvidas apenas quando um limite de seguranca e informado:

    limite_C          o proprio limite, para conferencia
    n_reprova_real    ensaios em que o componente de fato atinge o limite
    acerto_decisao_%  concordancia entre a decisao real e a prevista
    falso_aprovado    ensaios reprovados na realidade e aprovados pelo modelo
    falso_reprovado   ensaios aprovados na realidade e reprovados pelo modelo

    O R2 sai ausente quando o alvo assume um unico valor, situacao em que ele e
    indefinido e em que a implementacao do scikit-learn devolve zero, valor que se
    confunde com um resultado legitimo.
    """
    y_real, y_previsto = _valida(y_real, y_previsto)

    erro = y_previsto - y_real
    mae = float(np.mean(np.abs(erro)))

    # A raiz e calculada explicitamente: o parametro squared do mean_squared_error foi
    # removido do scikit-learn na versao 1.9, e o root_mean_squared_error so existe a
    # partir da 1.4. O calculo direto funciona em qualquer versao.
    rmse = float(np.sqrt(np.mean(erro ** 2)))

    if np.unique(y_real).size < 2:
        r2 = float("nan")
    else:
        ss_res = float(np.sum(erro ** 2))
        ss_tot = float(np.sum((y_real - np.mean(y_real)) ** 2))
        r2 = 1.0 - ss_res / ss_tot

    relativo = np.abs(erro) / y_real

    resultado = {
        "n": int(y_real.size),
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "vies": float(np.mean(erro)),
        "RMSE/MAE": (rmse / mae) if mae > 0 else float("nan"),
        "MAPE_%": 100.0 * float(np.mean(relativo)),
        "WAPE_%": 100.0 * float(np.sum(np.abs(erro)) / np.sum(y_real)),
        "dentro_10_%": 100.0 * float(np.mean(relativo <= MARGEM_KPI)),
    }

    if limite is not None and np.isfinite(limite):
        reprova_real = y_real >= limite
        reprova_prevista = y_previsto >= limite
        resultado.update({
            "limite_C": float(limite),
            "n_reprova_real": int(reprova_real.sum()),
            "acerto_decisao_%": 100.0 * float(np.mean(reprova_real == reprova_prevista)),
            "falso_aprovado": int(np.sum(reprova_real & ~reprova_prevista)),
            "falso_reprovado": int(np.sum(~reprova_real & reprova_prevista)),
        })

    return resultado


def consolidar(tabela, grupo="dataset",
               metricas=("MAE", "RMSE", "R2", "vies", "MAPE_%", "WAPE_%", "dentro_10_%")):
    """Consolida metricas por componente em metricas por area.

    A macro-media atribui o mesmo peso a cada componente e responde se o modelo funciona
    bem em todas as partes do produto. A micro-media pondera pelo numero de observacoes
    e responde qual o erro tipico de uma leitura qualquer.

    O R2 nao e comparavel entre areas, pois seu denominador e a variancia do alvo de
    cada area. Ele indica se o modelo supera a media dentro da propria area.
    """
    if grupo not in tabela.columns:
        raise ValueError(f"coluna de agrupamento ausente: {grupo}")
    if "n" not in tabela.columns:
        raise ValueError("a tabela precisa da coluna n para a micro-media")

    metricas = [m for m in metricas if m in tabela.columns]
    linhas = []

    for nome, bloco in tabela.groupby(grupo):
        linha = {grupo: nome, "componentes": len(bloco), "n_total": int(bloco["n"].sum())}
        for m in metricas:
            valores = bloco[m]
            validos = valores.notna()
            linha[f"{m}_macro"] = (float(valores[validos].mean())
                                   if validos.any() else float("nan"))
            if validos.any() and bloco.loc[validos, "n"].sum() > 0:
                linha[f"{m}_micro"] = float(
                    np.average(valores[validos], weights=bloco.loc[validos, "n"]))
            else:
                linha[f"{m}_micro"] = float("nan")

        # As metricas de decisao sao contagens, e por isso somadas em vez de promediadas
        for contagem in ("n_reprova_real", "falso_aprovado", "falso_reprovado"):
            if contagem in bloco.columns:
                linha[contagem] = int(bloco[contagem].fillna(0).sum())
        linhas.append(linha)

    return pd.DataFrame(linhas).sort_values(grupo).reset_index(drop=True)

In [ ]:
def mesclar(rf, gb, peso=0.5):
    """Combinacao ponderada das previsoes; peso e o do rf e (1 - peso) o do gb."""
    if not 0 <= peso <= 1:
        raise ValueError(f"peso fora de [0, 1]: {peso}")
    rf = np.asarray(rf, dtype=float)
    gb = np.asarray(gb, dtype=float)
    if rf.shape != gb.shape:
        raise ValueError(f"dimensoes diferentes: {rf.shape} e {gb.shape}")
    return peso * rf + (1 - peso) * gb

In [ ]:
rf, gb = previsoes["rf"].to_numpy(), previsoes["gb"].to_numpy()

verificacao = pd.Series({
    "peso 0,5 = media simples": np.allclose(mesclar(rf, gb, 0.5), (rf + gb) / 2),
    "peso 0 = gb": np.array_equal(mesclar(rf, gb, 0), gb),
    "peso 1 = rf": np.array_equal(mesclar(rf, gb, 1), rf),
})
assert verificacao.all()
verificacao

In [ ]:
def limite_de(col):
    lim = limites["limite_num"].get(col)
    return float(lim) if lim is not None and pd.notna(lim) else None


PESO = 0.5

linhas = []
for (sufixo, area, col), p in previsoes.groupby(["sufixo", "area", "componente"], sort=False):
    modelos = {"rf": p["rf"], "gb": p["gb"], "mescla": mesclar(p["rf"], p["gb"], PESO)}
    for modelo, previsto in modelos.items():
        linhas.append({"sufixo": sufixo, "area": area, "componente": col, "modelo": modelo,
                       **avaliar(p["real"], previsto, limite=limite_de(col))})

metricas = pd.DataFrame(linhas)
metricas.shape

In [ ]:
# consolidar agrupa por uma coluna so; sufixo e modelo entram pelo laco
blocos = []
for (sufixo, modelo), bloco in metricas.groupby(["sufixo", "modelo"], sort=False):
    blocos.append(consolidar(bloco, grupo="area").assign(sufixo=sufixo, modelo=modelo))

por_area = pd.concat(blocos, ignore_index=True)
por_area.shape

In [ ]:
MODELOS = ["rf", "gb", "mescla"]

comparacao = (por_area.pivot(index=["sufixo", "area"], columns="modelo",
                             values=["MAE_macro", "RMSE_macro", "R2_macro"])
              .rename(columns={"MAE_macro": "MAE", "RMSE_macro": "RMSE", "R2_macro": "R2"}, level=0))
comparacao = comparacao[[(m, c) for m in ("MAE", "RMSE", "R2") for c in MODELOS]]

# negativo quando a mesclagem erra menos que o melhor modelo individual
comparacao[("MAE", "mescla-melhor")] = (comparacao[("MAE", "mescla")]
                                        - comparacao["MAE"][["rf", "gb"]].min(axis=1))
comparacao.round(3)

In [ ]:
# Registro versionado da comparacao, ao lado do baseline de metricas por area
def pasta_documentos():
    for base in BASES + [os.path.join("..", "..", "..", "..")]:
        pasta = os.path.join(base, "documents", "extras", "relatorios_sprint_4")
        if os.path.isdir(pasta):
            return pasta
    raise FileNotFoundError("pasta documents/extras/relatorios_sprint_4 nao encontrada a partir de " + os.path.abspath("."))


def _num(v, sinal=False):
    if pd.isna(v):
        return "–"
    return f"{v:+.2f}".replace(".", ",") if sinal else f"{v:.2f}".replace(".", ",")


peso_txt = str(PESO).replace(".", ",")

cabecalho = ["Sufixo", "Área"] + [f"{m} {c}" for m in ("MAE", "RMSE", "R2") for c in MODELOS] + ["MAE mescla − melhor"]
linhas_md = []
for (sufixo, area), r in comparacao.iterrows():
    valores = [_num(r[(m, c)]) for m in ("MAE", "RMSE", "R2") for c in MODELOS]
    linhas_md.append("| " + " | ".join([sufixo, area] + valores + [_num(r[("MAE", "mescla-melhor")], sinal=True)]) + " |")

texto = "\n".join([
    "# Mesclagem RF e GB por média ponderada",
    "",
    f"Mesclagem das previsões fora-da-dobra de RandomForest e GradientBoosting por média ponderada com peso "
    f"{peso_txt} para o rf e {str(1 - PESO).replace('.', ',')} para o gb, componente a componente. "
    "A comparação é contra o baseline de métricas por área (`baseline_metricas_por_area.md`), com as mesmas previsões, "
    "a mesma célula canônica de métricas e a mesma validação nas 17 dobras leave-one-group-out. "
    "Gerado por `notebooks/colabs_S4/testes_extras/testes_treinamento/mesclagem_rf_gb.ipynb`.",
    "",
    "## Comparação por sufixo e área",
    "",
    "Médias macro (peso igual por componente). A última coluna é o MAE da mesclagem menos o menor MAE entre rf e gb; "
    "negativo quando a mesclagem erra menos que o melhor modelo individual.",
    "",
    "| " + " | ".join(cabecalho) + " |",
    "| " + " | ".join(["---"] * 2 + ["---:"] * (len(cabecalho) - 2)) + " |",
    *linhas_md,
    "",
    # texto fixo, escrito a partir da tabela acima; conferir se PESO ou as previsoes mudarem
    "Com peso 0,5, a mesclagem erra menos que o melhor modelo individual em mainback nos dois sufixos "
    "e em coil_temps no `_13cat`. Nas demais áreas piora contra o melhor individual: ct_knobs, ipc_comp, "
    "ipc_cooling, main_vent_sys e oven_hmi nos dois sufixos, e coil_temps no `_perfil`. A diferença vai "
    "de -0,26 °C em coil_temps (`_13cat`) a +0,21 °C em ipc_comp (`_13cat`).",
    "",
])

caminho_md = os.path.join(pasta_documentos(), "mesclagem_rf_gb_media.md")
with open(caminho_md, "w", encoding="utf-8", newline="\n") as f:
    f.write(texto)
caminho_md